# VSSD + First Impressions v2

Цель:
- использовать архитектуру **VSSD (Vision Mamba with Non-Causal State Space Duality)** как визуальный бэкбон;
- обучить видео-модель для регрессии по пяти персональным качествам (Big Five) на датасете **First Impressions v2**;
- подготовить базовые визуализации (распределение таргетов, кривые обучения, scatter-плоты предсказаний).

Датасет:
- структура: DATA/TRAIN, DATA/VALIDATION, DATA/TEST
- внутри каждой папки:
    - mp4-ролики
    - папка Annotation/ с двумя .pkl файлами:
        - аннотации (Big Five + interview) – dict of dicts
        - транскрипции – dict (video_name → текст)

Модель:
- VSSD используется как бэкбон для извлечения признаков из кадров (из статьи: некаузальный NC-SSD, глобальное состояние, инвариантность к направлению сканирования, сохранение 2D-структуры изображения).
- поверх бэкбона – простая временная агрегация и регрессионная голова на 5 признаков.


## 1. Базовая конфигурация и окружение

Здесь:
- задаём пути к данным,
- проверяем доступные GPU,
- включаем смешанную точность (fp16/bf16),
- закладываем возможность запуска в DDP (torchrun), но код будет работать и на одной GPU.


In [1]:
# %%
import os
import math
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler

from torchvision import transforms

import matplotlib.pyplot as plt

# Для воспроизводимости
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Пути к данным (подстрой под свою структуру)
DATA_ROOT = Path("../DATA")
TRAIN_DIR = DATA_ROOT / "TRAIN"
VAL_DIR = DATA_ROOT / "VALIDATION"
TEST_DIR = DATA_ROOT / "TEST"

assert TRAIN_DIR.exists(), f"Папка {TRAIN_DIR} не найдена"
assert VAL_DIR.exists(), f"Папка {VAL_DIR} не найдена"

# Проверим GPU
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name())


PyTorch version: 2.8.0+cpu
CUDA available: False
GPU count: 0


### Рекомендованная вычислительная конфигурация (для суперкомпьютера)

*Это текстовая рекомендация, не код (подстрой под свои кластеры):*

- 4–8× GPU уровня A100 / H100 / V100 32GB.
- Формат запуска: `torchrun --nproc_per_node=N ...`.
- Параметры:
  - разрешение кадров: 224×224 (как в ImageNet-конфигурациях VSSD);
  - число кадров на видео: 16–32;
  - batch size на GPU: 4–8 роликов (в зависимости от памяти);
  - mixed precision (`torch.cuda.amp.autocast`) + `GradScaler`.
- Для первых экспериментов можно начать с:
  - 1 GPU, 16 кадров, batch size = 2–4, чтобы отладить пайплайн.


## 2. Исследование файлов Annotation (.pkl)

В каждой фазе (train/val/test) внутри `Annotation/` лежат два файла:
- аннотации (Big Five + interview) – словарь словарей:
  - `annotation['extraversion'][video_name] -> float in [0, 1]`
  - `annotation['agreeableness'][video_name] -> float`
  - ...
  - `annotation['interview'][video_name] -> float`
- транскрипции – словарь:
  - `transcription[video_name] -> unicode строка`

Ниже – код, который:
- находит .pkl-файлы,
- загружает их,
- печатает примеры.


In [2]:
import pickle

def load_annotation_and_transcription(phase_dir: Path):
    annot_dir = phase_dir / "Annotation"
    assert annot_dir.exists(), f"Нет папки Annotation в {phase_dir}"

    pkl_files = list(annot_dir.glob("*.pkl"))
    assert len(pkl_files) == 2, f"Ожидалось 2 pkl-файла, найдено {len(pkl_files)} в {annot_dir}"

    # Определим, какой из файлов – аннотации, а какой – транскрипции
    def is_annotation_dict(obj):
        # простой эвристический тест структуры
        if not isinstance(obj, dict):
            return False
        # ключи верхнего уровня должны быть строками с именами признаков
        sample_keys = list(obj.keys())
        if not sample_keys:
            return False
        # значения – тоже словари
        return isinstance(obj[sample_keys[0]], dict)

    annotation = None
    transcription = None

    for pkl_path in pkl_files:
        with open(pkl_path, "rb") as f:
            obj = pickle.load(f, encoding="latin1")  # на всякий случай
        if is_annotation_dict(obj):
            annotation = obj
        else:
            transcription = obj

    assert annotation is not None, "Не удалось распознать файл с аннотациями"
    assert transcription is not None, "Не удалось распознать файл с транскрипциями"

    return annotation, transcription

train_annotation, train_transcription = load_annotation_and_transcription(TRAIN_DIR)
val_annotation, val_transcription = load_annotation_and_transcription(VAL_DIR)

print("Ключи верхнего уровня в annotation:", list(train_annotation.keys()))
some_trait = list(train_annotation.keys())[5]
print(f"Пример внутреннего словаря для признака '{some_trait}':")
items = list(train_annotation[some_trait].items())[:5]
for vid_name, value in items:
    print(vid_name, "->", value)

print("\nПримеры транскриптов:")
for vid_name, text in list(train_transcription.items())[:5]:
    print("Видео:", vid_name)
    print("Текст:", text[:120], "...")
    print("---")


Ключи верхнего уровня в annotation: ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'interview', 'openness']
Пример внутреннего словаря для признака 'openness':
J4GQm9j0JZ0.003.mp4 -> 0.4888888888888889
zEyRyTnIw5I.005.mp4 -> 0.3666666666666667
nskJh7v6v1U.004.mp4 -> 0.5111111111111111
6wHQsN5g2RM.000.mp4 -> 0.3777777777777778
dQOeQYWIgm8.000.mp4 -> 0.6222222222222222

Примеры транскриптов:
Видео: J4GQm9j0JZ0.003.mp4
Текст: He's cutting it and then turn around and see the end result, but I'm glad he didn't do that because I probably would've  ...
---
Видео: zEyRyTnIw5I.005.mp4
Текст: Responsibility to house the organ I had been given and I needed to tell them I was going to take good care of that organ ...
---
Видео: nskJh7v6v1U.004.mp4
Текст: I actually got quite a few sets of black pens this year, because I bought one pack. I think I bought two packs, actually ...
---
Видео: 6wHQsN5g2RM.000.mp4
Текст: I ate a lot. I'd like a lot of foods. I remember I have favor

### Список признаков (Big Five + interview)

Нормально проверить, что среди ключей аннотаций есть:
- `extraversion`, `agreeableness`, `conscientiousness`, `neuroticism`, `openness`, `interview`.


## 3. Подготовка видео: выбор кадров и преобразования

Для VSSD нам нужны "картинки" 3×224×224:
- выбираем фиксированное число кадров (например, 16) равномерно по ролику;
- применяем те же аугментации/нормировку, что и при обучении на ImageNet (минимальный вариант).


In [11]:
import cv2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

NUM_FRAMES = 16  # можно увеличить до 32 на мощном кластере

train_frame_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_frame_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def sample_indices(num_frames_total: int, num_samples: int) -> np.ndarray:
    """Равномерно выбираем num_samples индексов из диапазона [0, num_frames_total-1]."""
    if num_frames_total <= num_samples:
        # если кадров мало, просто повторяем последние
        indices = np.linspace(0, num_frames_total - 1, num_frames_total, dtype=int)
        pad = np.full(num_samples - num_frames_total, num_frames_total - 1, dtype=int)
        return np.concatenate([indices, pad])
    else:
        return np.linspace(0, num_frames_total - 1, num_samples, dtype=int)

def load_video_frames(
    video_path: Path,
    num_frames: int,
    transform,
) -> torch.Tensor:
    """
    Читает видео с помощью OpenCV, выбирает num_frames кадров, применяет transform к каждому.
    Возвращает тензор формы [T, C, H, W].
    """
    cap = cv2.VideoCapture(str(video_path))
    frames_bgr = []

    # читаем все кадры в список
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames_bgr.append(frame)
    cap.release()

    if len(frames_bgr) == 0:
        raise RuntimeError(f"Не удалось прочитать ни одного кадра из {video_path}")

    T = len(frames_bgr)
    indices = sample_indices(T, num_frames)

    frames = []
    for idx in indices:
        frame_bgr = frames_bgr[idx]              # H x W x 3, BGR
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)  # в RGB
        frame_tensor = transform(frame_rgb)      # ToPILImage внутри сам поймёт np.array
        frames.append(frame_tensor)

    frames = torch.stack(frames, dim=0)          # [T, C, H, W]
    return frames



## 4. Маппинг `video_name` → таргеты Big Five

Аннотации хранятся в виде:

```python
annotation['extraversion']['video_0001'] = 0.73
```

Для обучения нам нужен единый вектор таргетов:

```python
y = [extraversion, agreeableness, conscientiousness, neuroticism, openness]
```


In [12]:
BIG_FIVE_ORDER = [
    "extraversion",
    "agreeableness",
    "conscientiousness",
    "neuroticism",
    "openness",
]

def build_targets_dict(annotation: Dict[str, Dict[str, float]]) -> Dict[str, np.ndarray]:
    """
    Строит словарь:
      video_name -> np.array shape [5] (Big Five в фиксированном порядке)
    """
    targets = {}
    # возьмём список всех видео из одного из признаков
    some_trait = BIG_FIVE_ORDER[0]
    video_names = list(annotation[some_trait].keys())
    for vid in video_names:
        vals = []
        for trait in BIG_FIVE_ORDER:
            vals.append(annotation[trait][vid])
        targets[vid] = np.array(vals, dtype=np.float32)
    return targets

train_targets = build_targets_dict(train_annotation)
val_targets = build_targets_dict(val_annotation)

print("Пример таргета:")
for vid, y in list(train_targets.items())[:5]:
    print(vid, "->", y)


Пример таргета:
J4GQm9j0JZ0.003.mp4 -> [0.5233645  0.62637365 0.60194176 0.5520833  0.4888889 ]
zEyRyTnIw5I.005.mp4 -> [0.34579438 0.47252747 0.5825243  0.375      0.36666667]
nskJh7v6v1U.004.mp4 -> [0.25233644 0.4065934  0.4854369  0.29166666 0.51111114]
6wHQsN5g2RM.000.mp4 -> [0.45794392 0.50549453 0.39805827 0.48958334 0.37777779]
dQOeQYWIgm8.000.mp4 -> [0.60747665 0.4065934  0.6213592  0.48958334 0.62222224]


In [13]:
from typing import Dict, List, Tuple, Optional

class FirstImpressionsVideoDataset(Dataset):
    def __init__(
        self,
        phase_dir: Path,
        targets: Optional[Dict[str, np.ndarray]],
        num_frames: int = 16,
        train: bool = True,
    ):
        # базовые поля
        self.phase_dir = phase_dir
        self.targets = targets
        self.num_frames = num_frames
        self.train = train

        # собираем все mp4 в директории
        self.video_paths = sorted([p for p in phase_dir.glob("*.mp4")])
        assert self.video_paths, f"Не найдено mp4 в {phase_dir}"

        # список ID (как они называются в словаре targets)
        self.video_ids: List[str] = []

        self.frame_transform = train_frame_transform if train else eval_frame_transform

        if self.targets is not None:
            # фильтруем только те видео, у которых есть аннотация
            filtered_paths = []
            filtered_ids = []

            for p in self.video_paths:
                stem = p.stem          # без .mp4
                name = p.name          # с .mp4
                key = None

                # пробуем разные варианты названия
                if stem in self.targets:
                    key = stem
                elif name in self.targets:
                    key = name

                if key is not None:
                    filtered_paths.append(p)
                    filtered_ids.append(key)

            self.video_paths = filtered_paths
            self.video_ids = filtered_ids

            print(f"[{phase_dir.name}] всего mp4: {len(self.video_paths)}, "
                  f"с совпавшими аннотациями: {len(self.video_ids)}")

            # если после фильтрации не осталось видео — сразу падаем с понятным сообщением
            assert len(self.video_paths) > 0, (
                f"В {phase_dir} есть видео, но ни одно имя не совпало с аннотациями. "
                f"Проверь, как называются ключи в pkl и файлы .mp4"
            )
        else:
            # тестовый датасет без таргетов
            self.video_ids = [p.stem for p in self.video_paths]
            print(f"[{phase_dir.name}] видео (без таргетов): {len(self.video_paths)}")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx: int):
        # на всякий случай проверяем индекс
        if idx < 0 or idx >= len(self.video_paths):
            raise IndexError(f"Index {idx} out of range for dataset of size {len(self.video_paths)}")

        video_path = self.video_paths[idx]
        vid_id = self.video_ids[idx]  # ключ для словаря targets

        frames = load_video_frames(
            video_path=video_path,
            num_frames=self.num_frames,
            transform=self.frame_transform,
        )  # [T, C, H, W]

        if self.targets is None:
            target = np.zeros(len(BIG_FIVE_ORDER), dtype=np.float32)
        else:
            target = self.targets[vid_id]

        return {
            "video_name": vid_id,
            "frames": frames,
            "target": torch.from_numpy(target),
        }

train_ds = FirstImpressionsVideoDataset(TRAIN_DIR, train_targets, num_frames=NUM_FRAMES, train=True)
val_ds   = FirstImpressionsVideoDataset(VAL_DIR,  val_targets,   num_frames=NUM_FRAMES, train=False)

print("len(train_ds) =", len(train_ds))
print("len(val_ds)   =", len(val_ds))

sample = train_ds[0]
print("video_name:", sample["video_name"])
print("frames shape:", sample["frames"].shape)
print("target:", sample["target"])


[TRAIN] всего mp4: 6000, с совпавшими аннотациями: 6000
[VALIDATION] всего mp4: 2000, с совпавшими аннотациями: 2000
len(train_ds) = 6000
len(val_ds)   = 2000
video_name: --Ymqszjv54.001.mp4
frames shape: torch.Size([16, 3, 224, 224])
target: tensor([0.5514, 0.5275, 0.6505, 0.5000, 0.7444])


### DataLoader-ы

Для одиночной GPU – обычный `DataLoader`.
Для DDP – использовать `DistributedSampler` (см. комментарий).


In [ ]:
# %%
BATCH_SIZE = 2  # для отладки, на суперкомпьютере можно увеличить

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


## 6. Интеграция VSSD

Здесь важный момент.

В оригинальном репозитории VSSD:
- есть готовые конфиги и скрипты для **ImageNet / COCO / ADE20K**;:contentReference[oaicite:1]{index=1}
- бэкбон VSSD обучен на картинках формата 3×224×224.

Нам нужно:
- импортировать **предобученную** модель VSSD (Tiny/Small/Base);
- использовать её как **encoder** (feature extractor), без финального классификатора;
- поверх построить свою видео-голову.

Ниже – «обертка», которую нужно адаптировать под реальные классы в репозитории VSSD.

Вариант:
1. Клонируешь репозиторий `VSSD`:
   ```bash
   git clone https://github.com/YuHengsss/VSSD.git
   cd VSSD/classification
   pip install -r ../requirements.txt
   ```
2. В ноутбуке добавляешь путь `VSSD/classification` в `sys.path`.
3. Импортируешь нужный класс модели, например `VSSD_Tiny_iccv2025` (имя смотри в исходниках конфигов).


In [ ]:
import sys

# Путь к коду VSSD (подставь свой)
VSSD_CLASSIFICATION_ROOT = Path("/path/to/VSSD/classification")
assert VSSD_CLASSIFICATION_ROOT.exists(), "Укажи правильный путь к VSSD/classification"

if str(VSSD_CLASSIFICATION_ROOT) not in sys.path:
    sys.path.append(str(VSSD_CLASSIFICATION_ROOT))

# Пример: предположим, что внутри есть модуль models.vssd с классами VSSD_Tiny/Small/Base.
# Точные имена посмотри в репозитории и подправь импорт.
try:
    from models.vssd import VSSD_Tiny  # ПРИМЕР, проверь имя класса!
except ImportError as e:
    print("Не получилось импортировать VSSD_Tiny из models.vssd. "
          "Проверь структуру репозитория и подправь импорт.")
    raise e


### Обертка VSSDBackbone

Предполагаем, что:
- вызов `backbone(x)` возвращает признаки размерности `[B, D]` (или logits);
- у модели есть атрибут `num_features` или похожий для размерности выхода.

Если VSSD возвращает logits, нужно отбросить финальный `head` и использовать предпоследний слой.
Это уже зависит от конкретной реализации – здесь оставляю пометки.


In [ ]:
class VSSDBackbone(nn.Module):
    def __init__(self, pretrained_ckpt: Optional[str] = None):
        super().__init__()
        # Инициализируем VSSD Tiny (подстрой под свои конфиги)
        self.backbone = VSSD_Tiny()  # возможно, сюда нужно передать cfg

        # Если нужно – загрузим предобученные веса
        if pretrained_ckpt is not None:
            state = torch.load(pretrained_ckpt, map_location="cpu")
            # В репозитории VSSD чекпоинты могут хранить веса под ключом 'model' или 'state_dict'
            if "model" in state:
                self.backbone.load_state_dict(state["model"], strict=False)
            else:
                self.backbone.load_state_dict(state, strict=False)

        # Попытаемся определить размерность выходных признаков
        # Это зависимо от реализации; часто есть атрибут 'num_features'
        if hasattr(self.backbone, "num_features"):
            self.out_dim = self.backbone.num_features
        else:
            # на крайний случай сделаем один тестовый прогон
            with torch.no_grad():
                dummy = torch.randn(1, 3, 224, 224)
                feats = self._forward_features(dummy)
                self.out_dim = feats.shape[-1]

    def _forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """
        Прямой проход, возвращающий особенности (без финальной классификационной головы).
        Эту часть нужно подстроить под реальный интерфейс VSSD.
        """
        # В репозитории может быть метод forward_features() или похожий
        if hasattr(self.backbone, "forward_features"):
            feats = self.backbone.forward_features(x)
        else:
            # fallback – считаем, что forward возвращает logits, а мы выбрасываем final head заранее
            feats = self.backbone(x)
        return feats

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, 3, H, W]
        return: [B, D]
        """
        return self._forward_features(x)


## 7. Видео-модель для Big Five

Стандартный паттерн:
- вход: batch роликов `[B, T, C, H, W]`;
- объединяем по batch: `[B*T, C, H, W]` → прогоняем через VSSD;
- обратно собираем `[B, T, D]`;
- временная агрегация (например, среднее по времени);
- MLP-голова на 5 регрессионных выходов в [0, 1] (добавляем `Sigmoid`).


In [ ]:
class VSSDVideoRegressor(nn.Module):
    def __init__(self, backbone: VSSDBackbone, num_traits: int = 5, dropout: float = 0.1):
        super().__init__()
        self.backbone = backbone
        self.feature_dim = backbone.out_dim
        self.num_traits = num_traits

        # Простая временная агрегация: среднее по времени
        # Можно заменить на attention/RNN при необходимости.

        self.head = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, self.feature_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, num_traits),
            nn.Sigmoid(),  # чтобы явно получить значения в [0, 1]
        )

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        """
        frames: [B, T, C, H, W]
        return: [B, num_traits] – предсказания Big Five в [0,1]
        """
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)

        feats = self.backbone(x)          # [B*T, D]
        feats = feats.view(B, T, -1)      # [B, T, D]

        # Временная агрегация – среднее
        video_feats = feats.mean(dim=1)   # [B, D]

        out = self.head(video_feats)      # [B, num_traits]
        return out


## 8. Целевая функция и метрики

Для Big Five обычно используют:
- MSE / MAE;
- корреляцию Пирсона / R².

Здесь сделаем:
- loss = MSE;
- метрики: MAE и корреляция Пирсона по каждой из 5 шкал.

In [ ]:
# %%
from torch.cuda.amp import autocast, GradScaler

def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> Dict[str, float]:
    """
    pred, target: [N, 5] в [0,1]
    """
    pred_np = pred.detach().cpu().numpy()
    targ_np = target.detach().cpu().numpy()

    mae_per_trait = np.mean(np.abs(pred_np - targ_np), axis=0)

    # корреляция Пирсона
    corr_per_trait = []
    for i in range(pred_np.shape[1]):
        x = pred_np[:, i]
        y = targ_np[:, i]
        if np.std(x) < 1e-6 or np.std(y) < 1e-6:
            corr = 0.0
        else:
            corr = np.corrcoef(x, y)[0, 1]
        corr_per_trait.append(corr)

    metrics = {}
    for i, trait in enumerate(BIG_FIVE_ORDER):
        metrics[f"MAE_{trait}"] = float(mae_per_trait[i])
        metrics[f"Corr_{trait}"] = float(corr_per_trait[i])

    metrics["MAE_mean"] = float(mae_per_trait.mean())
    metrics["Corr_mean"] = float(np.mean(corr_per_trait))
    return metrics


### Циклы train / validate

Используем:
- `torch.cuda.amp.autocast` для mixed precision;
- `GradScaler` для стабильного обучения.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    device: torch.device,
) -> Tuple[float, Dict[str, float]]:
    model.train()
    total_loss = 0.0
    all_pred = []
    all_targ = []

    for batch in loader:
        frames = batch["frames"].to(device, non_blocking=True)  # [B, T, C, H, W]
        target = batch["target"].to(device, non_blocking=True)  # [B, 5]

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            pred = model(frames)            # [B, 5]
            loss = nn.MSELoss()(pred, target)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * frames.size(0)
        all_pred.append(pred.detach())
        all_targ.append(target.detach())

    all_pred = torch.cat(all_pred, dim=0)
    all_targ = torch.cat(all_targ, dim=0)

    epoch_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(all_pred, all_targ)
    return epoch_loss, metrics

def validate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[float, Dict[str, float]]:
    model.eval()
    total_loss = 0.0
    all_pred = []
    all_targ = []

    with torch.no_grad():
        for batch in loader:
            frames = batch["frames"].to(device, non_blocking=True)
            target = batch["target"].to(device, non_blocking=True)

            with autocast():
                pred = model(frames)
                loss = nn.MSELoss()(pred, target)

            total_loss += loss.item() * frames.size(0)
            all_pred.append(pred.detach())
            all_targ.append(target.detach())

    all_pred = torch.cat(all_pred, dim=0)
    all_targ = torch.cat(all_targ, dim=0)

    epoch_loss = total_loss / len(loader.dataset)
    metrics = compute_metrics(all_pred, all_targ)
    return epoch_loss, metrics

## 9. Запуск обучения

Для начала – прототип на одной GPU (или CPU, если нужно только проверить код).

На кластере:
- увеличиваешь `EPOCHS`, `BATCH_SIZE`, `NUM_FRAMES`;
- переходишь на запуск через `torchrun` и DistributedDataParallel.

In [ ]:
# %%
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Инициализация бэкбона VSSD
# Укажи путь к предобученному чекпоинту, если есть (или оставь None для обучения с нуля)
PRETRAINED_CKPT = None  # пример: "/path/to/vssd_tiny_iccv2025.pth"

backbone = VSSDBackbone(pretrained_ckpt=PRETRAINED_CKPT)
model = VSSDVideoRegressor(backbone=backbone, num_traits=len(BIG_FIVE_ORDER))
model = model.to(device)

# Оптимизатор
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = GradScaler(enabled=torch.cuda.is_available())

EPOCHS = 5  # для отладки; на суперкомпьютере можно 30–100

history = {
    "train_loss": [],
    "val_loss": [],
    "val_corr_mean": [],
    "val_mae_mean": [],
}

for epoch in range(1, EPOCHS + 1):
    print(f"\n=== Эпоха {epoch}/{EPOCHS} ===")

    train_loss, train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, device)
    print(f"Train loss: {train_loss:.4f}, MAE_mean: {train_metrics['MAE_mean']:.4f}, Corr_mean: {train_metrics['Corr_mean']:.4f}")

    val_loss, val_metrics = validate(model, val_loader, device)
    print(f"Val   loss: {val_loss:.4f}, MAE_mean: {val_metrics['MAE_mean']:.4f}, Corr_mean: {val_metrics['Corr_mean']:.4f}")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_corr_mean"].append(val_metrics["Corr_mean"])
    history["val_mae_mean"].append(val_metrics["MAE_mean"])

    # Сохранение чекпоинта
    ckpt_path = Path("checkpoints")
    ckpt_path.mkdir(exist_ok=True, parents=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "history": history,
        },
        ckpt_path / f"vssd_first_impressions_epoch{epoch:03d}.pth",
    )


## 10. Визуализация: кривые обучения и распределение таргетов

1. train/val loss по эпохам
2. средняя корреляция на валидации
3. гистограммы реальных значений Big Five


In [ ]:
# %%
# 1) Кривые loss
plt.figure(figsize=(6, 4))
plt.plot(history["train_loss"], label="train_loss")
plt.plot(history["val_loss"], label="val_loss")
plt.xlabel("Эпоха")
plt.ylabel("Loss (MSE)")
plt.title("Кривые обучения (MSE)")
plt.legend()
plt.grid(True)
plt.show()

# 2) Средняя корреляция
plt.figure(figsize=(6, 4))
plt.plot(history["val_corr_mean"], label="val_corr_mean")
plt.xlabel("Эпоха")
plt.ylabel("Средняя корреляция Пирсона")
plt.title("Качество на валидации (Corr_mean)")
plt.legend()
plt.grid(True)
plt.show()

# 3) Распределение таргетов по Big Five (train+val)
all_y = np.concatenate(
    [np.stack(list(train_targets.values()), axis=0),
     np.stack(list(val_targets.values()), axis=0)],
    axis=0
)

plt.figure(figsize=(12, 4))
for i, trait in enumerate(BIG_FIVE_ORDER):
    plt.subplot(1, len(BIG_FIVE_ORDER), i+1)
    plt.hist(all_y[:, i], bins=20, alpha=0.8)
    plt.title(trait)
    plt.xlabel("значение")
plt.tight_layout()
plt.show()


## 11. Scatter-плоты: предсказания vs ground truth

Для лучшего понимания, где модель ошибается, можно визуализировать:
- scatter `y_true` vs `y_pred` для каждого из пяти признаков.

In [ ]:
# %%
# Соберём предсказания на валидации
model.eval()
all_pred = []
all_targ = []

with torch.no_grad():
    for batch in val_loader:
        frames = batch["frames"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)
        with autocast():
            pred = model(frames)
        all_pred.append(pred.cpu())
        all_targ.append(target.cpu())

all_pred = torch.cat(all_pred, dim=0).numpy()
all_targ = torch.cat(all_targ, dim=0).numpy()

plt.figure(figsize=(15, 3))
for i, trait in enumerate(BIG_FIVE_ORDER):
    plt.subplot(1, len(BIG_FIVE_ORDER), i+1)
    plt.scatter(all_targ[:, i], all_pred[:, i], alpha=0.3, s=8)
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel("GT")
    plt.ylabel("Pred")
    plt.title(trait)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
plt.tight_layout()
plt.show()
